# 02 - Prepare workforce capacity

This notebook prepares the 2025 officer/staff capacity used by the allocation.

The current model uses the whole wider function `Local policing`.

In [10]:
import pandas as pd
from pathlib import Path
import sqlite3

## Settings

In [11]:
DATA_DIR = Path("../../data")
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"

WORKFORCE_YEAR = 2025
CAPACITY_VERSION = "local_policing"
WIDER_FUNCTION_NAME = "local policing"

## Load and clean workforce file

In [12]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    workforce = pd.read_sql_query(
        "SELECT * FROM police_workforce_resources_v1;",
        conn,
    )

workforce_2025 = workforce[workforce["year"] == WORKFORCE_YEAR].copy()
workforce_2025["wider_function_name_clean"] = workforce_2025["wider_function_name"].str.lower()

workforce_2025.head()

,year,pfa_code,pfa_name,region,worker_type,ethnicity_5_1,ethnicity_3_1,sex,function_subgroup_number,function_subgroup_name,wider_function_number,wider_function_name,frontline_type,fte,wider_function_name_clean
70793,2025,E23000036,Avon and Somerset,South West,Police Community Support Officer,Mixed,Ethnic minorities,Female,1a,Neighbourhood Policing,1,Local policing,Visible operational front line,1.0,local policing
70794,2025,E23000036,Avon and Somerset,South West,Police Community Support Officer,Not stated,Not stated,Female,1a,Neighbourhood Policing,1,Local policing,Visible operational front line,11.0,local policing
70795,2025,E23000036,Avon and Somerset,South West,Police Community Support Officer,Other ethnic group,Ethnic minorities,Female,1a,Neighbourhood Policing,1,Local policing,Visible operational front line,1.0,local policing
70796,2025,E23000036,Avon and Somerset,South West,Police Community Support Officer,White,White,Female,1a,Neighbourhood Policing,1,Local policing,Visible operational front line,95.0,local policing
70797,2025,E23000036,Avon and Somerset,South West,Police Community Support Officer,Asian or Asian British,Ethnic minorities,Male,1a,Neighbourhood Policing,1,Local policing,Visible operational front line,3.0,local policing


In [13]:
print("missing FTE values:", workforce_2025["fte"].isna().sum())

missing FTE values: 15


## Select workforce capacity

In [14]:
selected_workforce = workforce_2025[
    workforce_2025["wider_function_name_clean"] == WIDER_FUNCTION_NAME.lower()
].copy()

capacity_filter_description = "Wider function: Local policing"

print("capacity version:", CAPACITY_VERSION)
print("filter:", capacity_filter_description)
print("selected rows:", len(selected_workforce))
print("selected total FTE:", selected_workforce["fte"].sum())

capacity version: local_policing
filter: Wider function: Local policing
selected rows: 1726
selected total FTE: 66837.0


In [15]:
selected_workforce["capacity_group"] = selected_workforce["worker_type"].apply(
    lambda x: "pcso_fte" if x == "Police Community Support Officer" else "regular_police_staff_fte"
)

force_capacity_long = (
    selected_workforce
    .groupby(["pfa_code", "pfa_name", "capacity_group"], as_index=False)
    .agg(fte=("fte", "sum"))
)

force_capacity_model = (
    force_capacity_long
    .pivot_table(
        index=["pfa_code", "pfa_name"],
        columns="capacity_group",
        values="fte",
        fill_value=0,
    )
    .reset_index()
)
force_capacity_model.columns.name = None

for col in ["pcso_fte", "regular_police_staff_fte"]:
    if col not in force_capacity_model.columns:
        force_capacity_model[col] = 0

force_capacity_model["total_capacity"] = (
    force_capacity_model["pcso_fte"]
    + force_capacity_model["regular_police_staff_fte"]
)

force_capacity_model.head()

,pfa_code,pfa_name,pcso_fte,regular_police_staff_fte,total_capacity
0,E23000001,Metropolitan Police,1263.0,12499.0,13762.0
1,E23000002,Cumbria,34.0,515.0,549.0
2,E23000003,Lancashire,176.0,1333.0,1509.0
3,E23000004,Merseyside,165.0,1611.0,1776.0
4,E23000005,Greater Manchester,265.0,3313.0,3578.0


## Check which forces are currently in the model area

The capacity table is not filtered here. Forces outside the current forecast/geography are kept, but marked with `in_current_model_area`.

In [16]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    msoa_context = pd.read_sql_query("SELECT * FROM msoa_context_v1;", conn)

model_pfa_codes = set(msoa_context["pfa_code"].drop_duplicates())

force_capacity_model["in_current_model_area"] = (
    force_capacity_model["pfa_code"].isin(model_pfa_codes)
)

force_capacity_model.sort_values(
    ["in_current_model_area", "pfa_code"],
    ascending=[False, True],
)

,pfa_code,pfa_name,pcso_fte,regular_police_staff_fte,total_capacity,in_current_model_area
0,E23000001,Metropolitan Police,1263.0,12499.0,13762.0,True
1,E23000002,Cumbria,34.0,515.0,549.0,True
2,E23000003,Lancashire,176.0,1333.0,1509.0,True
3,E23000004,Merseyside,165.0,1611.0,1776.0,True
4,E23000005,Greater Manchester,265.0,3313.0,3578.0,True
5,E23000006,Cheshire,85.0,1015.0,1100.0,True
6,E23000007,Northumbria,78.0,1746.0,1824.0,True
7,E23000008,Durham,122.0,644.0,766.0,True
8,E23000009,North Yorkshire,102.0,648.0,750.0,True
9,E23000010,West Yorkshire,476.0,2485.0,2961.0,True


In [17]:
print("forces in capacity table:", len(force_capacity_model))
print("forces currently in model area:", force_capacity_model["in_current_model_area"].sum())
print("forces not currently in model area:", (~force_capacity_model["in_current_model_area"]).sum())
print("total capacity:", force_capacity_model["total_capacity"].sum())

force_capacity_model.loc[
    ~force_capacity_model["in_current_model_area"],
    ["pfa_code", "pfa_name", "total_capacity"],
]

forces in capacity table: 43
forces currently in model area: 39
forces not currently in model area: 4
total capacity: 66837.0


,pfa_code,pfa_name,total_capacity
39,W15000001,North Wales,1035.0
40,W15000002,Gwent,785.0
41,W15000003,South Wales,1763.0
42,W15000004,Dyfed-Powys,638.0


## Save capacity table

In [18]:
capacity_settings = pd.DataFrame(
    [
        {
            "workforce_year": WORKFORCE_YEAR,
            "capacity_version": CAPACITY_VERSION,
            "capacity_filter_description": capacity_filter_description,
            "wider_function_name": WIDER_FUNCTION_NAME,
        }
    ]
)

with sqlite3.connect(ALLOC_DB_PATH) as conn:
    force_capacity_model.to_sql("force_capacity_v1", conn, if_exists="replace", index=False)
    capacity_settings.to_sql("capacity_settings_v1", conn, if_exists="replace", index=False)

print("Saved force_capacity_v1 rows:", len(force_capacity_model))
print("Capacity version:", CAPACITY_VERSION)
print("Filter:", capacity_filter_description)

Saved force_capacity_v1 rows: 43
Capacity version: local_policing
Filter: Wider function: Local policing
